# 📬 Pulso Académico SIIMAG — v5 (Portada Metro)

Boletín trimestral con diseño tipo portada del periódico Metro CDMX.

**Novedades v5:**
- Diseño portada Metro: rojo `#E31E24` + azul `#0033A0` + negro `#1A1A1A`
- Titulares con doble sentido generados por Gemini, uno por categoría
- Bajada periodística breve por sección
- Asunto del correo con gancho Metro
- Semáforo correcto por categoría

**Compatible con:** Gmail (Google Workspace) ✅

In [ ]:
# ── 1. SETUP ────────────────────────────────────────────────────────────────
!pip install git+https://github.com/claudiodanielpc-ag/cd_base.git -q
!pip install unidecode -q

from cd_base import ConexionBD
from google.colab import drive, ai
drive.mount('/content/drive')

import pandas as pd
import json, re, unidecode, smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

print('✅ Librerías listas')

In [ ]:
# ── 2. CONEXIÓN A BD ─────────────────────────────────────────────────────────
ruta   = '/content/drive/MyDrive/credenciales/bd_produccion.txt'
bd     = ConexionBD(ruta)
engine = bd.conectar('migracion_aws_do')

In [ ]:
# ── 3. PARÁMETROS DEL PERIODO ─────────────────────────────────────────────────
PERIODO_ACTUAL_INI = '2026-01-01'
PERIODO_ACTUAL_FIN = '2026-03-31'
PERIODO_COMP_INI   = '2025-01-01'
PERIODO_COMP_FIN   = '2025-03-31'
PERIODO_LABEL      = 'Enero – Marzo 2026'
PERIODO_COMP_LABEL = 'Enero – Marzo 2025'
PERIODO_CORTO      = 'ENE–MAR 2026'
ANIO_COMP          = '2025'

LOGO_URL = 'https://raw.githubusercontent.com/claudiodanielpc-ag/siimag/refs/heads/main/logo_siimag_nuevo_transp.png'

In [ ]:
# ── 4. EXTRACCIÓN DE INDICADORES ─────────────────────────────────────────────
conn   = engine.raw_connection()
cursor = conn.cursor()

cursor.callproc(
    'sp_tbl_indicadores_general_programas',
    (1, None, None,
     PERIODO_ACTUAL_INI, PERIODO_ACTUAL_FIN,
     PERIODO_COMP_INI,   PERIODO_COMP_FIN)
)

resultados = []
while True:
    rows = cursor.fetchall()
    if rows:
        cols = [col[0] for col in cursor.description]
        resultados.append(pd.DataFrame(rows, columns=cols))
    if not cursor.nextset():
        break
cursor.close()

indicadores = resultados[0]
print(f'✅ {len(indicadores)} indicadores cargados')
indicadores

In [ ]:
# ── 4-ALT. DATOS DE EJEMPLO (descomenta si no tienes BD) ─────────────────────

# indicadores = pd.DataFrame([
#   {'id_indicador':3,  'nombre':'Inscritos',            'porcentual':0,'categoria':1,'total':570,  'total_comp':641,   'porcentaje':-11.08,'tendencia':'down','descripcion':'Alumnos dados de alta en el programa académico.'},
#   {'id_indicador':4,  'nombre':'Inscritos facturados', 'porcentual':0,'categoria':1,'total':558,  'total_comp':542,   'porcentaje': 2.95, 'tendencia':'up',  'descripcion':'Inscritos con al menos una materia cargada.'},
#   {'id_indicador':6,  'nombre':'Cargas',               'porcentual':0,'categoria':1,'total':15854,'total_comp':16854, 'porcentaje':-5.93, 'tendencia':'down','descripcion':'Materias inscritas durante el periodo.'},
#   {'id_indicador':9,  'nombre':'Eficiencia terminal',  'porcentual':1,'categoria':1,'total':30.37,'total_comp':25.51, 'porcentaje':19.05, 'tendencia':'up',  'descripcion':'Porcentaje de alumnos que concluyeron materias.'},
#   {'id_indicador':5,  'nombre':'Reactivados',          'porcentual':0,'categoria':2,'total':137,  'total_comp':321,   'porcentaje':-57.32,'tendencia':'down','descripcion':'Alumnos que retomaron su trayectoria.'},
#   {'id_indicador':12, 'nombre':'Tasa de reactivación', 'porcentual':1,'categoria':2,'total':63.72,'total_comp':126.88,'porcentaje':-49.78,'tendencia':'down','descripcion':'Porcentaje de alumnos que retoman vs. bajas.'},
#   {'id_indicador':10, 'nombre':'Reinscritos',          'porcentual':0,'categoria':2,'total':158,  'total_comp':160,   'porcentaje':-1.25, 'tendencia':'down','descripcion':'Alumnos que retomaron tras baja de empresa.'},
#   {'id_indicador':7,  'nombre':'Tasa de reinserción',  'porcentual':1,'categoria':2,'total':31.35,'total_comp':30.36, 'porcentaje': 3.26, 'tendencia':'up',  'descripcion':'Porcentaje que vuelve a inscribirse.'},
#   {'id_indicador':1,  'nombre':'Bajas de la empresa',  'porcentual':0,'categoria':3,'total':504,  'total_comp':527,   'porcentaje':-4.36, 'tendencia':'down','descripcion':'Alumnos que dejaron de laborar en la empresa.'},
#   {'id_indicador':2,  'nombre':'Bajas del programa',   'porcentual':0,'categoria':3,'total':202,  'total_comp':187,   'porcentaje': 8.02, 'tendencia':'up',  'descripcion':'Alumnos que dejaron el programa pero siguen en empresa.'},
#   {'id_indicador':8,  'nombre':'Tasa de Deserción',    'porcentual':1,'categoria':3,'total':36.20,'total_comp':34.50, 'porcentaje': 4.93, 'tendencia':'up',  'descripcion':'Porcentaje que no continúa el programa.'},
# ])
# print('✅ Datos de ejemplo listos')

In [ ]:
# ── 5. TEXTOS CON GEMINI — ESTILO METRO ──────────────────────────────────────
#
# El periódico Metro CDMX usa titulares cortos con doble sentido:
# el titular dice una cosa pero insinúa otra sin explicarlo.
# "Es Tepito juguetón" = celebración del Día del Niño / picardía.
# "¡A ver, golondrinas!" = alumnos que se van y no regresan / canción.
# La gracia está en que el lector tarda un segundo en caerle.
#
# REGLAS para los titulares:
# - Máx 6 palabras por titular
# - Doble sentido con vocabulario académico (inscribir, cargar, baja,
#   terminal, reactivar, desertar, reinsertar, terminar, entrar, salir)
# - SIN explicar el chiste. El titular va solo.
# - SIN afirmar records históricos (no "nunca", no "el peor", no "el mejor")
# - La bajada sí puede ser directa y periodística (2 oraciones máx)

resumen_cat1 = indicadores[indicadores['categoria']==1][['nombre','total','total_comp','porcentaje']].to_string(index=False)
resumen_cat2 = indicadores[indicadores['categoria']==2][['nombre','total','total_comp','porcentaje']].to_string(index=False)
resumen_cat3 = indicadores[indicadores['categoria']==3][['nombre','total','total_comp','porcentaje']].to_string(index=False)

prompt_textos = f"""
Eres redactor del periódico Metro de la Ciudad de México.
Tu especialidad: titulares cortos con doble sentido académico.
El chiste está en que el titular dice una cosa pero insinúa otra — sin explicarlo.
Ejemplos del estilo deseado:
  - "Pocos entraron, pero los que entraron... terminaron" (entrar/terminar = académico + picardía)
  - "Les rogaron que volvieran y ni así" (reactivar alumnos + rogarle a alguien)
  - "Se dieron permiso solos" (darse de baja + tomarse el día sin avisar)

Genera un JSON con exactamente estas claves:
{{
  "titular_cap": "<titular captación, máx 8 palabras, doble sentido con entrar/inscribir/cargar/terminar>",
  "bajada_cap": "<bajada periodística directa, máx 2 oraciones, con los datos clave>",
  "titular_ret": "<titular retención, máx 8 palabras, doble sentido con volver/regresar/reactivar/reinsertar>",
  "bajada_ret": "<bajada periodística directa, máx 2 oraciones>",
  "titular_baj": "<titular bajas, máx 8 palabras, doble sentido con irse/baja/desertar/salir/permiso>",
  "bajada_baj": "<bajada periodística directa, máx 2 oraciones>",
  "asunto": "<asunto del correo, máx 10 palabras, con doble sentido suave, que dé ganas de abrir>"
}}

REGLAS ESTRICTAS:
- Sin explicar el chiste. El titular va solo.
- Sin afirmar récords (no uses: nunca, jamás, el peor, el mejor, histórico)
- Sin groserías. Sin emojis. Solo texto.
- Devuelve SOLO el JSON, sin backticks ni texto extra.

Datos Captación ({PERIODO_LABEL} vs {PERIODO_COMP_LABEL}):
{resumen_cat1}

Datos Retención:
{resumen_cat2}

Datos Bajas y deserción:
{resumen_cat3}
"""

raw = ai.generate_text(prompt_textos).strip()
# Limpiar backticks por si Gemini los agrega
raw = re.sub(r'^```json|^```|```$', '', raw, flags=re.MULTILINE).strip()
textos = json.loads(raw)

titular_cap = textos['titular_cap'].upper()
bajada_cap  = textos['bajada_cap']
titular_ret = textos['titular_ret'].upper()
bajada_ret  = textos['bajada_ret']
titular_baj = textos['titular_baj'].upper()
bajada_baj  = textos['bajada_baj']
asunto_correo = textos['asunto']

print('📰 TITULARES GENERADOS:')
print(f'  Captación : {titular_cap}')
print(f'  Retención : {titular_ret}')
print(f'  Bajas     : {titular_baj}')
print(f'  Asunto    : {asunto_correo}')

In [ ]:
# ── 6. FUNCIONES AUXILIARES ───────────────────────────────────────────────────

ROJO      = '#E31E24'
AZUL      = '#0033A0'
NEGRO     = '#1A1A1A'
ROJO_OSC  = '#B01018'
FONDO     = '#f0f0f0'
SEP       = '#dddddd'
VERDE     = '#388E3C'
VERDE_NUM = '#1a6b1a'
ROJO_NUM  = '#D32F2F'
GRIS      = '#888888'
GRIS_NUM  = '#555555'
BG_VERDE  = '#e8f5e9'
TX_VERDE  = '#2E7D32'
BG_ROJO   = '#ffebee'
TX_ROJO   = '#C62828'

F  = "font-family:'Century Gothic','Barlow Condensed',Arial,sans-serif;"
FB = "font-family:'Barlow Condensed',Arial,sans-serif;"
FM = "font-family:monospace;"

def fmt_valor(row):
    v = row['total']
    if row['porcentual'] == 1:
        return f"{v:,.1f}", '%'
    elif v >= 1000:
        return f"{int(v):,}", ''
    return f"{int(v)}", ''

def fmt_comp(row):
    v = row['total_comp']
    if row['porcentual'] == 1:
        return f"{v:,.1f}%"
    elif v >= 1000:
        return f"{int(v):,}"
    return f"{int(v)}"

def es_bueno(row):
    sube = row['porcentaje'] > 0
    return (not sube) if row['categoria'] == 3 else sube

def color_num(row):
    if row['porcentaje'] == 0: return GRIS_NUM
    return VERDE_NUM if es_bueno(row) else ROJO_NUM

def color_barra(row):
    if row['porcentaje'] == 0: return GRIS
    return VERDE if es_bueno(row) else ROJO_NUM

def badge_html(row):
    sube = row['porcentaje'] > 0
    pct  = abs(row['porcentaje'])
    flec = '&#9650;' if sube else '&#9660;'
    bg, tx = (BG_VERDE, TX_VERDE) if es_bueno(row) else (BG_ROJO, TX_ROJO)
    return (f'<span style="display:inline-block;background:{bg};color:{tx};'
            f'padding:2px 6px;border-radius:3px;font-size:10px;font-weight:700;{FB}">'
            f'{flec} {pct:.1f}%</span>')

print('✅ Funciones listas')

In [ ]:
# ── 7. GENERADORES DE BLOQUES HTML ────────────────────────────────────────────

def kpi_grande(row):
    """KPI centrado para grilla de 4 (Captación)."""
    valor, sufijo = fmt_valor(row)
    cn   = color_num(row)
    badg = badge_html(row)
    sup  = f'<sup style="font-size:12px;">{sufijo}</sup>' if sufijo else ''
    return f"""
    <td style="background:#ffffff;padding:10px 8px;text-align:center;vertical-align:top;">
      <div style="font-size:26px;font-weight:900;line-height:1;color:{cn};
                  margin-bottom:3px;{FB}">{valor}{sup}</div>
      <div style="font-size:9px;letter-spacing:1px;color:#aaa;text-transform:uppercase;
                  margin-bottom:5px;{FB}">{row['nombre']}</div>
      {badg}
    </td>"""


def kpi_horizontal(row):
    """KPI con barra lateral (Retención y Bajas, grilla 2)."""
    valor, sufijo = fmt_valor(row)
    comp  = fmt_comp(row)
    cn    = color_num(row)
    cb    = color_barra(row)
    sube  = row['porcentaje'] > 0
    flec  = '&#9650;' if sube else '&#9660;'
    pct   = abs(row['porcentaje'])
    cv    = VERDE if es_bueno(row) else ROJO_NUM
    sup   = f'<span style="font-size:12px;color:#ccc;">{sufijo}</span>' if sufijo else ''
    return f"""
    <td style="background:#ffffff;padding:12px 14px;vertical-align:top;">
      <table cellpadding="0" cellspacing="0" width="100%">
        <tr>
          <td width="3" style="background:{cb};border-radius:2px;">&nbsp;</td>
          <td style="padding-left:10px;">
            <div style="font-size:24px;font-weight:900;line-height:1;
                        margin-bottom:2px;color:{cn};{FB}">{valor}{sup}</div>
            <div style="font-size:9px;letter-spacing:1px;color:#bbb;
                        text-transform:uppercase;margin-bottom:5px;{FB}">{row['nombre']}</div>
            <div style="font-size:11px;font-weight:700;color:{cv};{FB}">{flec} {pct:.1f}%</div>
            <div style="font-size:10px;color:#ccc;margin-top:2px;{F}">vs. {comp} en {ANIO_COMP}</div>
          </td>
        </tr>
      </table>
    </td>"""


def titular_seccion(kicker, kicker_color, titular, color1, color2, color3, bajada, borde_color):
    """Bloque titular estilo portada Metro."""
    palabras = titular.split()
    mitad    = len(palabras) // 2
    linea1   = ' '.join(palabras[:mitad])
    linea2   = ' '.join(palabras[mitad:])
    return f"""
    <tr>
      <td colspan="99" style="padding:12px 16px 0;background:#ffffff;">
        <div style="font-size:10px;letter-spacing:2.5px;color:{kicker_color};
                    text-transform:uppercase;font-weight:700;
                    margin-bottom:4px;{FB}">{kicker}</div>
        <div style="font-size:52px;font-weight:900;line-height:.9;
                    text-transform:uppercase;margin-bottom:8px;{FB}">
          <span style="color:{color1};">{linea1}</span><br>
          <span style="color:{color2};">{linea2}</span>
        </div>
        <div style="font-size:13px;color:#444;line-height:1.5;{F}
                    border-left:4px solid {borde_color};padding-left:10px;
                    margin-bottom:12px;">
          {bajada}
        </div>
      </td>
    </tr>"""


def sec_bar(label, bg):
    return f"""
    <tr>
      <td colspan="99" style="background:{bg};padding:5px 14px;">
        <table width="100%" cellpadding="0" cellspacing="0"><tr>
          <td style="font-size:9px;letter-spacing:2px;color:rgba(255,255,255,.5);
                     text-transform:uppercase;{FB}">{label}</td>
          <td align="right" style="font-size:9px;letter-spacing:2px;
                     color:rgba(255,255,255,.5);text-transform:uppercase;{FB}">{PERIODO_CORTO}</td>
        </tr></table>
      </td>
    </tr>"""


def grilla(filas_html, cols):
    return f"""
    <tr>
      <td colspan="99" style="padding:0;">
        <table width="100%" cellpadding="0" cellspacing="1"
          style="background:{SEP};">
          {filas_html}
        </table>
      </td>
    </tr>"""


def filas_de_a(cat_df, fn, n_cols):
    html = ''
    items = list(cat_df.iterrows())
    for i in range(0, len(items), n_cols):
        grupo = items[i:i+n_cols]
        celdas = ''.join(fn(r) for _, r in grupo)
        if len(grupo) < n_cols:
            celdas += f'<td style="background:#fff;"></td>' * (n_cols - len(grupo))
        html += f'<tr>{celdas}</tr>'
    return html

print('✅ Generadores listos')

In [ ]:
# ── 8. CONSTRUCCIÓN DEL HTML ──────────────────────────────────────────────────

cat1 = indicadores[indicadores['categoria']==1].reset_index(drop=True)
cat2 = indicadores[indicadores['categoria']==2].reset_index(drop=True)
cat3 = indicadores[indicadores['categoria']==3].reset_index(drop=True)

html = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width,initial-scale=1.0">
  <title>Pulso Académico SIIMAG · {PERIODO_LABEL}</title>
</head>
<body style="margin:0;padding:0;background:{FONDO};{F}">

<table width="100%" cellpadding="0" cellspacing="0"
  style="background:{FONDO};padding:16px 0;">
<tr><td align="center">

<table width="620" cellpadding="0" cellspacing="0"
  style="background:#ffffff;max-width:620px;
         border:1px solid {SEP};overflow:hidden;">

  <!-- CABECERA ROJA ESTILO METRO -->
  <tr>
    <td colspan="99" style="background:{ROJO};padding:0;">
      <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
          <td style="padding:8px 16px 6px;">
            <img src="{LOGO_URL}" style="height:26px;width:auto;
                 filter:brightness(0) invert(1);opacity:.95;" alt="SIIMAG">
          </td>
          <td align="right" style="padding:8px 16px 6px;">
            <div style="font-size:10px;color:rgba(255,255,255,.45);
                        letter-spacing:1px;text-transform:uppercase;{FB}">Academia Global</div>
            <div style="font-size:14px;color:#fff;font-weight:700;
                        letter-spacing:.5px;{FB}">Pulso Académico SIIMAG</div>
          </td>
        </tr>
        <tr>
          <td colspan="2" style="background:{ROJO_OSC};padding:4px 16px;">
            <table width="100%" cellpadding="0" cellspacing="0"><tr>
              <td style="font-size:10px;letter-spacing:1.5px;
                         color:rgba(255,255,255,.5);
                         text-transform:uppercase;{FB}">Reporte trimestral</td>
              <td align="center" style="font-size:10px;letter-spacing:1.5px;
                         color:rgba(255,255,255,.5);
                         text-transform:uppercase;{FB}">{PERIODO_LABEL}</td>
              <td align="right" style="font-size:10px;letter-spacing:1.5px;
                         color:rgba(255,255,255,.5);
                         text-transform:uppercase;{FB}">vs. {PERIODO_COMP_LABEL}</td>
            </tr></table>
          </td>
        </tr>
      </table>
    </td>
  </tr>

  <!-- FRANJA ROJO+AZUL -->
  <tr><td colspan="99" style="height:5px;
    background:linear-gradient(90deg,{ROJO} 50%,{AZUL} 50%);"></td></tr>

  <!-- CAPTACIÓN: titular -->
  {titular_seccion('Captación del trimestre', ROJO, titular_cap,
                   NEGRO, ROJO, AZUL, bajada_cap, ROJO)}

  <!-- CAPTACIÓN: datos -->
  {sec_bar('Captación', AZUL)}
  {grilla(filas_de_a(cat1, kpi_grande, 4), 4)}

  <!-- RETENCIÓN: titular -->
  {titular_seccion('Retención', AZUL, titular_ret,
                   AZUL, ROJO, NEGRO, bajada_ret, AZUL)}

  <!-- RETENCIÓN: datos -->
  {sec_bar('Retención', NEGRO)}
  {grilla(filas_de_a(cat2, kpi_horizontal, 2), 2)}

  <!-- BAJAS: titular -->
  {titular_seccion('Bajas y deserción', ROJO, titular_baj,
                   NEGRO, AZUL, ROJO, bajada_baj, ROJO)}

  <!-- BAJAS: datos -->
  {sec_bar('Bajas y deserción', ROJO)}
  {grilla(filas_de_a(cat3, kpi_horizontal, 2), 2)}

  <!-- FOOTER AZUL -->
  <tr>
    <td colspan="99" style="background:{AZUL};padding:10px 16px;">
      <table width="100%" cellpadding="0" cellspacing="0"><tr>
        <td style="font-size:10px;color:rgba(255,255,255,.5);
                   letter-spacing:1px;text-transform:uppercase;{FB}">
          Sistema de Información, Inteligencia y Monitoreo · Academia Global
        </td>
        <td align="right">
          <a href="https://erp.agcollege.com.mx/#/login"
             style="font-size:11px;color:{ROJO};letter-spacing:2px;
                    text-transform:uppercase;text-decoration:none;
                    font-weight:700;{FB}">SIIMAG &#8594;</a>
        </td>
      </tr></table>
    </td>
  </tr>

</table>
</td></tr>
</table>

</body>
</html>"""

print(f'✅ HTML generado — {len(html):,} caracteres')

In [ ]:
# ── 9. PREVIEW EN COLAB ─────────────────────────────────────────────────────
from IPython.display import HTML
HTML(html)

In [ ]:
# ── 10. ENVÍO POR CORREO ──────────────────────────────────────────────────────
# El asunto lo genera Gemini con estilo Metro en la celda 5.
# Para sobreescribirlo manualmente descomenta la línea de abajo:
# asunto_correo = f'Pulso Académico SIIMAG · {PERIODO_LABEL}'

lista_correos = [
    'nelson.amparan@academiaglobal.mx',
    'karla.garzon@academiaglobal.mx',
    'jazmin.garzon@academiaglobal.mx',
    'javier.cazarez@academiaglobal.mx',
    'rosario.verduzco@academiaglobal.mx',
    'rodolfo.castro@academiaglobal.mx',
    'elsie.garzon@academiaglobal.mx',
    'vanessa.fuentes@academiaglobal.mx',
    'andreyely.ronquillo@academiaglobal.mx',
    'fernanda.castro@academiaglobal.mx',
    'nery.perez@academiaglobal.mx',
    'ernesto.torres@academiaglobal.mx',
    'anabelen.avila@academiaglobal.mx',
    'erik.velasco@academiaglobal.mx',
    'claudio.pacheco@academiaglobal.mx',
    'almendra.navidad@academiaglobal.mx',
    'abdiel.gutierrez@academiaglobal.mx',
    'leticia.beltran@academiaglobal.mx',
    'fedra.llanes@academiaglobal.mx',
    'analaura.roman@academiaglobal.mx',
    'mayra.alvarez@academiaglobal.mx',
]

REMITENTE    = 'siimag@academiaglobal.mx'
APP_PASSWORD = 'REEMPLAZA_CON_TU_NUEVA_APP_PASSWORD'

msg = MIMEMultipart('alternative')
msg['Subject'] = asunto_correo
msg['From']    = REMITENTE
msg['Bcc']     = ', '.join(lista_correos)
msg.attach(MIMEText(html, 'html'))

with smtplib.SMTP('smtp.gmail.com', 587) as server:
    server.starttls()
    server.login(REMITENTE, APP_PASSWORD)
    server.send_message(msg)

print(f'✅ Correo enviado a {len(lista_correos)} destinatarios')
print(f'📨 Asunto: {asunto_correo}')